# Claude Code vs GitHub Copilot by Model

Run-level resolution summaries for `Claude Code` and `GitHub Copilot` on `opus-4.5` and `opus-4.6`.

In [42]:
import sys
from pathlib import Path

# Add notebooks root to path for utils import
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from utils import compute_pass_metrics, load_aggregate_results

MODELS = ["opus-4.5", "opus-4.6"]
AGENTS = ["GitHub Copilot", "Claude Code"]
MODEL_SETUPS = {
    "opus-4.5": {
        "GitHub Copilot": "copilot-opus-4-5",
        "Claude Code": "claude-opus-4-5",
    },
    "opus-4.6": {
        "GitHub Copilot": "copilot-opus-4-6",
        "Claude Code": "claude-opus-4-6",
    },
}
COLORS = {
    "GitHub Copilot": "steelblue",
    "Claude Code": "coral",
}

all_results = load_aggregate_results(category="bug-fix")
selected_results = {model: {agent: all_results[setup_name].copy() for agent, setup_name in MODEL_SETUPS[model].items()} for model in MODELS}

In [43]:
import plotly.graph_objects as go
from IPython.display import Markdown, display

from bcbench.results.metrics import bootstrap_ci


def build_mean_ci_table(model: str) -> pd.DataFrame:
    rows = []
    for agent in AGENTS:
        setup_df = selected_results[model][agent]
        per_run_means = setup_df.groupby("run_id")["resolved"].mean().to_numpy(dtype=float) * 100
        boot = bootstrap_ci(per_run_means)
        rows.append(
            {
                "Agent": agent,
                "Runs": len(per_run_means),
                "Mean %": boot["mean"],
                "CI Low": boot["ci_low"] if boot["ci_low"] is not None else boot["mean"],
                "CI High": boot["ci_high"] if boot["ci_high"] is not None else boot["mean"],
            }
        )

    return pd.DataFrame(rows).round(3)


def plot_mean_ci(model: str) -> go.Figure:
    mean_ci_df = build_mean_ci_table(model)
    fig = go.Figure()
    for _, row in mean_ci_df.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row["Mean %"]],
                y=[row["Agent"]],
                mode="markers",
                marker={"size": 14, "color": COLORS[row["Agent"]]},
                error_x={
                    "type": "data",
                    "symmetric": False,
                    "array": [row["CI High"] - row["Mean %"]],
                    "arrayminus": [row["Mean %"] - row["CI Low"]],
                    "thickness": 2,
                    "width": 10,
                },
                name=row["Agent"],
            )
        )

    x_min = max(0, float(mean_ci_df["CI Low"].min()) - 5)
    x_max = min(100, float(mean_ci_df["CI High"].max()) + 5)

    fig.update_layout(
        title=f"Mean Resolution Rate with 95% Bootstrap CI ({model})",
        xaxis_title="Resolution Rate (%)",
        xaxis={"range": [x_min, x_max], "dtick": 5, "showgrid": True, "gridcolor": "lightgray"},
        yaxis_title="",
        height=320,
        showlegend=False,
        margin={"l": 140},
    )
    return fig


def build_pass_hat_5_table(model: str) -> pd.DataFrame:
    rows = []
    for agent in AGENTS:
        metrics = compute_pass_metrics(selected_results[model][agent], k=5)
        rows.append({"Agent": agent, "pass^5 %": metrics["pass_hat_k"] * 100})

    return pd.DataFrame(rows).round(1)


def render_model_section(model: str) -> None:
    fig = plot_mean_ci(model)
    fig.show()
    display(Markdown("**Mean Resolution Rate**"))
    display(build_mean_ci_table(model))
    display(Markdown("**pass^5**"))
    display(build_pass_hat_5_table(model))

## 1. Opus 4.5

Mean resolution rate with bootstrap confidence intervals and `pass^5` for the `opus-4.5` runs.

In [44]:
render_model_section("opus-4.5")

**Mean Resolution Rate**

,Agent,Runs,Mean %,CI Low,CI High
0,GitHub Copilot,5,58.416,56.436,60.198
1,Claude Code,5,57.426,55.050,59.208


**pass^5**

,Agent,pass^5 %
0,GitHub Copilot,38.6
1,Claude Code,31.7


## 2. Opus 4.6

Mean resolution rate with bootstrap confidence intervals and `pass^5` for the `opus-4.6` runs.

In [45]:
render_model_section("opus-4.6")

**Mean Resolution Rate**

,Agent,Runs,Mean %,CI Low,CI High
0,GitHub Copilot,5,65.149,62.772,67.723
1,Claude Code,5,68.515,65.743,71.287


**pass^5**

,Agent,pass^5 %
0,GitHub Copilot,50.5
1,Claude Code,49.5


## 3. Statistical Check

Run-level difference summary for each model.

Reported quantities: `5` runs per agent, observed difference in mean run-level resolution rate (`Claude Code - GitHub Copilot`), bootstrap 95% interval, and exact permutation p-value.

In [ ]:
from itertools import combinations

import numpy as np

rng = np.random.default_rng(42)
difference_rows = []

for model in MODELS:
    copilot = selected_results[model]["GitHub Copilot"].groupby("run_id")["resolved"].mean().to_numpy(dtype=float) * 100
    claude = selected_results[model]["Claude Code"].groupby("run_id")["resolved"].mean().to_numpy(dtype=float) * 100
    observed_diff = float(claude.mean() - copilot.mean())

    bootstrap_diffs = []
    for _ in range(20000):
        claude_sample = rng.choice(claude, size=len(claude), replace=True)
        copilot_sample = rng.choice(copilot, size=len(copilot), replace=True)
        bootstrap_diffs.append(float(claude_sample.mean() - copilot_sample.mean()))

    ci_low, ci_high = np.percentile(bootstrap_diffs, [2.5, 97.5])

    pooled = np.concatenate([claude, copilot])
    permutation_diffs = []
    for chosen_indices in combinations(range(len(pooled)), len(claude)):
        chosen = set(chosen_indices)
        group_a = np.array([pooled[index] for index in range(len(pooled)) if index in chosen], dtype=float)
        group_b = np.array([pooled[index] for index in range(len(pooled)) if index not in chosen], dtype=float)
        permutation_diffs.append(float(group_a.mean() - group_b.mean()))

    permutation_diffs = np.array(permutation_diffs)
    p_value = float(np.mean(np.abs(permutation_diffs) >= abs(observed_diff)))

    difference_rows.append(
        {
            "Model": model,
            "Runs per agent": len(claude),
            "Observed diff in mean run-level rate (Claude - Copilot)": observed_diff,
            "95% bootstrap CI low": ci_low,
            "95% bootstrap CI high": ci_high,
            "Exact permutation p-value": p_value,
        }
    )

pd.DataFrame(difference_rows).round(3)